In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from warnings import filterwarnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.neural_network import MLPClassifier

from utils import get_data, train_and_plot_learning_curves

seed = 42
wine_X_tr, wine_X_val, wine_X_test, wine_y_tr, wine_y_val, wine_y_test = get_data(seed)


filterwarnings("ignore", category=ConvergenceWarning)

In [19]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted
from sklearn.discriminant_analysis import unique_labels


class NN(BaseEstimator, ClassifierMixin):
    def __init__(self, hidden_layer_sizes=[100], dropout=0.0, epochs=200, batch_size=128, 
                 early_stopping=True, patience=10, tol=1e-4, random_state=seed, **kwargs):
        self.hidden_layer_sizes = hidden_layer_sizes
        self.dropout = dropout
        self.epochs = epochs
        self.batch_size = batch_size
        self.early_stopping = early_stopping
        self.patience = patience
        self.tol = tol
        self.random_state = random_state
        self.kwargs = kwargs
        for key, value in kwargs.items():
            setattr(self, key, value)
        
    def fit(self, X, y):
        torch.manual_seed(self.random_state)
        X, y = check_X_y(X, y)
        self.classes_ = unique_labels(y)
        self.num_features_ = X.shape[1]
        
        y_mapped = np.searchsorted(self.classes_, y)

        layer_dims = [self.num_features_, *self.hidden_layer_sizes, len(self.classes_)]
        layers = []
        for i in range(len(layer_dims)-2):
            layers.append(nn.Linear(layer_dims[i], layer_dims[i+1]))
            layers.append(nn.ReLU())
            if self.dropout:
                layers.append(nn.Dropout(self.dropout))
        layers.append(nn.Linear(layer_dims[-2], layer_dims[-1]))

        self.model_ = nn.Sequential(*layers)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(self.model_.parameters(), **self.kwargs)

        self.model_.train()
        X_tensor = torch.as_tensor(X, dtype=torch.float32)
        y_tensor = torch.as_tensor(y_mapped, dtype=torch.long)
        
        n_samples = len(X)
        best_loss = float('inf')
        no_improve = 0

        for epoch in range(self.epochs):
            perm = torch.randperm(n_samples)
            X_shuffled = X_tensor[perm]
            y_shuffled = y_tensor[perm]
            
            epoch_loss = 0.0
            for i in range(0, n_samples, self.batch_size):
                batch_X = X_shuffled[i : i + self.batch_size]
                batch_y = y_shuffled[i : i + self.batch_size]
                optimizer.zero_grad()
                outputs = self.model_(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
            
            if self.early_stopping:
                if epoch_loss + self.tol < best_loss:
                    best_loss = epoch_loss
                    no_improve = 0
                else:
                    no_improve += 1
                    if no_improve >= self.patience:
                        break

        return self

    def predict_proba(self, X):
        check_is_fitted(self)
        X = check_array(X)

        self.model_.eval()
        with torch.inference_mode():
            X_tensor = torch.as_tensor(X, dtype=torch.float32)
            outputs = self.model_(X_tensor)
            probs = torch.softmax(outputs, dim=1)

        return probs.numpy()
    
    def predict(self, X):
        probs = self.predict_proba(X)
        return self.classes_[np.argmax(probs, axis=1)]


In [20]:
train_and_plot_learning_curves(
    [
        NN(hidden_layer_sizes=(16,)),
        NN(hidden_layer_sizes=(32,)),
        NN(hidden_layer_sizes=(64,)),
        NN(hidden_layer_sizes=(128,)),
        NN(hidden_layer_sizes=(16, 16)),
        NN(hidden_layer_sizes=(32, 32)),
        NN(hidden_layer_sizes=(64, 64)),
        NN(hidden_layer_sizes=(128, 128)),
    ],
    wine_X_tr,
    wine_y_tr,
    wine_X_val,
    wine_y_val,
    seed,
    "hidden_layer_sizes"
)
train_and_plot_learning_curves(
    [
        MLPClassifier(hidden_layer_sizes=(16,)),
        MLPClassifier(hidden_layer_sizes=(32,)),
        MLPClassifier(hidden_layer_sizes=(64,)),
        MLPClassifier(hidden_layer_sizes=(128,)),
        MLPClassifier(hidden_layer_sizes=(16, 16)),
        MLPClassifier(hidden_layer_sizes=(32, 32)),
        MLPClassifier(hidden_layer_sizes=(64, 64)),
        MLPClassifier(hidden_layer_sizes=(128, 128)),
    ],
    wine_X_tr,
    wine_y_tr,
    wine_X_val,
    wine_y_val,
    seed,
    "hidden_layer_sizes"
)

In [21]:
train_and_plot_learning_curves(
    [
        NN(hidden_layer_sizes=(64,), lr=0.0001),
        NN(hidden_layer_sizes=(64,), lr=0.0005),
        NN(hidden_layer_sizes=(64,), lr=0.001),
        NN(hidden_layer_sizes=(64,), lr=0.005),
        NN(hidden_layer_sizes=(64,), lr=0.01),
        NN(hidden_layer_sizes=(64,), lr=0.05),
        NN(hidden_layer_sizes=(64,), lr=0.1),
        NN(hidden_layer_sizes=(64,), lr=0.5),
        NN(hidden_layer_sizes=(64,), lr=1),
    ],
    wine_X_tr,
    wine_y_tr,
    wine_X_val,
    wine_y_val,
    seed,
    "lr"
)


train_and_plot_learning_curves(
    [
        MLPClassifier(hidden_layer_sizes=(64,), learning_rate_init=0.0001),
        MLPClassifier(hidden_layer_sizes=(64,), learning_rate_init=0.0005),
        MLPClassifier(hidden_layer_sizes=(64,), learning_rate_init=0.001),
        MLPClassifier(hidden_layer_sizes=(64,), learning_rate_init=0.005),
        MLPClassifier(hidden_layer_sizes=(64,), learning_rate_init=0.01),
        MLPClassifier(hidden_layer_sizes=(64,), learning_rate_init=0.05),
        MLPClassifier(hidden_layer_sizes=(64,), learning_rate_init=0.1),
        MLPClassifier(hidden_layer_sizes=(64,), learning_rate_init=0.5),
        MLPClassifier(hidden_layer_sizes=(64,), learning_rate_init=1),
    ],
    wine_X_tr,
    wine_y_tr,
    wine_X_val,
    wine_y_val,
    seed,
    "learning_rate_init"
)

In [22]:
train_and_plot_learning_curves(
    [
        NN(weight_decay=0, hidden_layer_sizes=(64,)),
        NN(weight_decay=0.00001, hidden_layer_sizes=(64,)),
        NN(weight_decay=0.00005, hidden_layer_sizes=(64,)),
        NN(weight_decay=0.0001, hidden_layer_sizes=(64,)),
        NN(weight_decay=0.0005, hidden_layer_sizes=(64,)),
        NN(weight_decay=0.001, hidden_layer_sizes=(64,)),
        NN(weight_decay=0.005, hidden_layer_sizes=(64,)),
        NN(weight_decay=0.01, hidden_layer_sizes=(64,)),
        NN(weight_decay=0.05, hidden_layer_sizes=(64,)),
        NN(weight_decay=0.1, hidden_layer_sizes=(64,)),
    ],
    wine_X_tr,
    wine_y_tr,
    wine_X_val,
    wine_y_val,
    seed,
    "weight_decay"
)


train_and_plot_learning_curves(
    [
        MLPClassifier(alpha=0, hidden_layer_sizes=(64,)),
        MLPClassifier(alpha=0.00001, hidden_layer_sizes=(64,)),
        MLPClassifier(alpha=0.00005, hidden_layer_sizes=(64,)),
        MLPClassifier(alpha=0.0001, hidden_layer_sizes=(64,)),
        MLPClassifier(alpha=0.0005, hidden_layer_sizes=(64,)),
        MLPClassifier(alpha=0.001, hidden_layer_sizes=(64,)),
        MLPClassifier(alpha=0.005, hidden_layer_sizes=(64,)),
        MLPClassifier(alpha=0.01, hidden_layer_sizes=(64,)),
        MLPClassifier(alpha=0.05, hidden_layer_sizes=(64,)),
        MLPClassifier(alpha=0.1, hidden_layer_sizes=(64,)),
    ],
    wine_X_tr,
    wine_y_tr,
    wine_X_val,
    wine_y_val,
    seed,
    "alpha"
)

In [23]:
train_and_plot_learning_curves(
    [
        NN(dropout=0, hidden_layer_sizes=(64,)),
        NN(dropout=0.1, hidden_layer_sizes=(64,)),
        NN(dropout=0.2, hidden_layer_sizes=(64,)),
        NN(dropout=0.3, hidden_layer_sizes=(64,)),
        NN(dropout=0.4, hidden_layer_sizes=(64,)),
        NN(dropout=0.5, hidden_layer_sizes=(64,)),
        NN(dropout=0.6, hidden_layer_sizes=(64,)),
        NN(dropout=0.7, hidden_layer_sizes=(64,)),
    ],
    wine_X_tr,
    wine_y_tr,
    wine_X_val,
    wine_y_val,
    seed,
    "dropout"
)

In [ ]:
best_model = NN(weight_decay=0.001, hidden_layer_sizes=(64,))
best_model.fit(wine_X_tr, wine_y_tr)
best_model.score(wine_X_test, wine_y_test)